# Coverage matching


Computational scripts save assignment `-1` first, then random assignments `0–99`.
`PAIRING_VIEW` changes only the loaded/displayed results. Full results exclude `-1`
and give each random assignment equal weight after averaging its internal folds.
Mean ± SD are descriptive, not confidence intervals. Missing assignments are an
error; assignment zero and incomplete batches are never substituted for the full result.
Raw signal preparation runs once in a terminal and includes every eligible MEG subject.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT.parent), str(ROOT.parent/'LB')]
PAIRING_VIEW = 'full'  # 'matched': -1; 'full': mean ± SD of IDs 0–99.
N_PAIRINGS = 100
PAIRING_SEED = 2026
from vs0_run_pairings import activate, format_command, pairing_context
activate(PAIRING_VIEW, N_PAIRINGS, PAIRING_SEED, display_only=True)
import vs0_load_and_config as shared_data
from vs0_pairing_results import config_view, path_view, show, safe_load, ResultView
ANALYSIS_OUTPUT_ROOT = shared_data.analysis_output_root(ROOT)
DISPLAY_COMPONENTS = 5
SEED = 2026  # Trial/source-control seed, independent of PAIRING_SEED.
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'
MAX_GRAM_GIB = 2.
from vs1_coverage_figures import apply_style, MODALITY_COLORS, SETUP_COLORS
apply_style()

OUTPUT_DIR = ANALYSIS_OUTPUT_ROOT/'coverage'
from vs1_run_coverage import load_datasets, prepare_analysis
from vs1_coverage_matching_utils import variance_summary
from vs1_elect_number_effect import load_elect_number_effect
from vs1_coverage_figures import plot_elect_number_effect

## Configuration and terminal preparation

In [ ]:
MEG_DIR = shared_data.MEG_DATA_DIR
IEEG_DIR = ROOT/'ieeg_shortWOBS_fs250'
METADATA_CSV = None
PROJECT_PATH = None
SFREQ = 250.
MEG_TMIN = None
MEG_TIMES_FILE = None
CORRECT_MEG_ZSCORE = True
CORRECT_IEEG_X1000 = True
CONDITIONS = [1,2]
CONFIG = dict(data=dict(meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR),
    metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
    project_path=str(PROJECT_PATH) if PROJECT_PATH else None,
    meg_coordinate_unit='m', ieeg_coordinate_unit='mm', meg_tmin=MEG_TMIN,
    meg_scaling='channel_zscore' if CORRECT_MEG_ZSCORE else 'none',
    ieeg_scaling='none', ieeg_multiplier=1000. if CORRECT_IEEG_X1000 else 1.,
    meg_times_file=str(MEG_TIMES_FILE) if MEG_TIMES_FILE else None,
    include_all_meg=True, sfreq=SFREQ, conditions=CONDITIONS,
    prepared_cache_dir=str(ROOT/'out'/'prepared_data')),
    output_dir=str(OUTPUT_DIR), block_scaling=BLOCK_SCALING, condition_mode=CONDITION_MODE,
    n_components=N_COMPONENTS, seed=SEED, pairing_seed=PAIRING_SEED,
    n_pairings=N_PAIRINGS, max_gram_gib=MAX_GRAM_GIB)

CONFIG.update(channel_step=100, channel_repeats=20, stability_repeats=N_PAIRINGS)

In [ ]:
# This cell writes configuration and prints a terminal command; no raw signals are prepared here.
CONFIG_FILE = OUTPUT_DIR/'config.json'
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2))
import shlex
print(shlex.join(['python','-u',str(ROOT/'vs0_prepare_data.py'),'--config',str(CONFIG_FILE)]))

## Load the configured result view

Signal datasets are loaded only for the matched descriptive coverage displays. Each section below loads its saved computation using `CONFIG` and `PAIRING_VIEW`; full results average fitted outputs, not input signals.

In [ ]:
with pairing_context(-1, PAIRING_SEED):
    datasets = safe_load(load_datasets, CONFIG)
ieeg = datasets['iEEG'] if datasets is not None else None

## Metadata, variance, feature counts, and brain coverage

Always uses the matched assignment (-1), regardless of PAIRING_VIEW. Non-pairwise MEG constructions use the complete eligible pool.

In [ ]:
if datasets is not None:
    MAX_COVERAGE_POINTS = 20000

    summary = variance_summary(datasets)
    for name, data in datasets.items():
        n_subjects = (len(data.source_data['ieeg_subjects']) if name == 'iEEG'
                      else len(data.pairing) if name in ('paired_coverage', 'random_control')
                      else len(data.source_data['meg_subjects']))
        summary.loc[name, 'n_subjects'] = n_subjects
        summary.loc[name, 'n_unique_locations'] = len(data.metadata[['x', 'y', 'z']].drop_duplicates())
        summary.loc[name, 'scale'] = data.scale
        summary.loc[name, 'feature_type'] = 'electrode' if name == 'iEEG' else 'source'
    display(summary)

In [ ]:
if datasets is not None:
    from nilearn import plotting
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    for data in datasets.values():
        has_meg = 'meg_subject' in data.metadata.columns
        if has_meg:
            df = data.metadata[['x', 'y', 'z', 'meg_subject']].drop_duplicates()
            all_subjects = sorted(df['meg_subject'].dropna().unique())
            subject_cmap = plt.get_cmap('viridis', len(all_subjects))
            node_cmap = ListedColormap([subject_cmap(i) for i in range(len(all_subjects))])
            node_vmin = -0.5
            node_vmax = len(all_subjects) - 0.5

        else:
            df = data.metadata[['x', 'y', 'z']].drop_duplicates()
            node_cmap = 'Greys'
            node_vmin = 0
            node_vmax = 1

        if len(df) > MAX_COVERAGE_POINTS:df = df.iloc[np.linspace(0, len(df) - 1, MAX_COVERAGE_POINTS, dtype=int)]
        pos = df[['x', 'y', 'z']].to_numpy()
        if has_meg:subject_ids = df['meg_subject'].map({s: i for i, s in enumerate(all_subjects)}).to_numpy()
        else:subject_ids = np.ones(len(pos))
        fig = plt.figure(figsize=(11, 3.5))
        plotting.plot_markers(subject_ids, pos,node_size=8,node_cmap=node_cmap,node_vmin=node_vmin,node_vmax=node_vmax,node_threshold=None,figure=fig,colorbar=False,title=f'{data.name}: {data.n_features:,} features / {len(pos):,} locations',)

        for ax in fig.axes:
            ax.grid(True, alpha=.25)

## Direct MEG comparisons by manipulation

All manipulation metrics, spectra, and effective dimensionalities are shown. Pairing-dependent constructions vary across assignments; non-pairwise constructions are unchanged and have zero assignment SD.

In [ ]:
RECOMPUTE = False
prepare_analysis(CONFIG, 'manipulations', CONFIG_FILE, RECOMPUTE)
information = safe_load(config_view, CONFIG, 'manipulations', PAIRING_VIEW, N_PAIRINGS)

In [ ]:
if information is not None:
    show(information, max_components=DISPLAY_COMPONENTS, tables=True, arrays=True)

## MEG source-count effect

Computed once using all eligible MEG participants; independent of the pairing view. Source-subset repetitions remain separate from subject assignments.

In [ ]:
directory = prepare_analysis(CONFIG, 'channels', CONFIG_FILE, RECOMPUTE)
channel_effect = load_elect_number_effect(directory) if directory is not None else None
if channel_effect is not None:
    plot_elect_number_effect(channel_effect, min_channels=None, max_channels=10000, figsize=(8,8))
    plt.show()

## PCA for all datasets

Time courses and coverage-slot weights are aligned to the saved matched fit using temporal scores only. Variance spectra and PCA comparison metrics retain native rank. Every exported correlation and variance metric is loaded. Full spatial summaries describe fixed iEEG coverage slots, not identical native MEG sources.

In [ ]:
prepare_analysis(CONFIG, 'pca', CONFIG_FILE, RECOMPUTE)
pca_view = safe_load(config_view, CONFIG, 'pca', PAIRING_VIEW, N_PAIRINGS)

In [ ]:
if pca_view is not None:
    show(pca_view, max_components=DISPLAY_COMPONENTS, tables=True, arrays=True)

## Stability under random subject reassignment

Full view only. The terminal computation includes the matched reference (-1) plus exactly N_PAIRINGS assignments from the full eligible pool. Between-assignment pairs share fits and are not independent observations.

In [ ]:
if PAIRING_VIEW == 'full':
    from vs1_coverage_stability import load_stability_results
    directory = prepare_analysis(CONFIG, 'stability', CONFIG_FILE, RECOMPUTE)
    stability = load_stability_results(directory, expected_experiment='pairing') if directory is not None else None
else:
    stability = None
    print('Reassignment stability is displayed only in full view.')

In [ ]:
if stability is not None:
    pairs = pd.concat([stability['component_pairs'], pd.read_csv(directory/'full_component_pairs.csv')], ignore_index=True)
    pairs = pairs[pairs['repeat'] >= 0]  # Exclude the matched reference from random summaries.
    display(pairs.groupby(['domain','comparison','dataset','component_a']).abs_r.agg(['mean','std','count']))
    for (domain,comparison),frame in pairs.groupby(['domain','comparison']):
        fig,ax=plt.subplots(figsize=(9,4))
        for dataset,group in frame.groupby('dataset'):
            values=group.groupby('component_a').abs_r.agg(['mean','std'])
            ax.errorbar(values.index,values['mean'],yerr=values['std'],label=dataset,capsize=3)
        ax.set(title=f'{domain}: {comparison}; random assignments only',xlabel='Component',ylabel='Matched |r|')
        ax.legend();plt.show()